Teacher model PTQ; 
Student model QAT

In [ ]:
import numpy as np
import time
import tensorflow as tf
import keras
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, MultiHeadAttention, LayerNormalization, TimeDistributed, Flatten
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import KLDivergence, CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
import tensorflow_model_optimization as tfmot

 Copy of data_loading_preprocessing.ipynb
Run the script first to load the samples before executing other functions or scripts

In [ ]:
# Learning Rate Scheduler
def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler = LearningRateScheduler(scheduler)

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)

In [ ]:
# MultiHeadAttention Layer
# Create positional encoding
def positional_encoding(seq_len, d_model):
    angles = 1 / np.power(10000., (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    positional_encoding = np.zeros((seq_len, d_model))
    positional_encoding[:, 0::2] = np.sin(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 0::2]))
    positional_encoding[:, 1::2] = np.cos(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 1::2]))
    positional_encoding = positional_encoding[np.newaxis, ...]
    return tf.cast(positional_encoding, dtype=tf.float32)

#Create MultiHeadAttention Encoder

class ScaledDotProductAttention(Layer):
    def __init__(self, dropout_ratio=0.1, **kwargs):
        super(ScaledDotProductAttention, self).__init__(**kwargs)
        self.dropout = Dropout(dropout_ratio)

    def call(self, q, k, v, mask=None, training=False):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)
        output = tf.matmul(attention_weights, v)
        return output



class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads, dropout_ratio=0.1, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0  # d_model must be divisible by num_heads

        self.depth = d_model // self.num_heads
        self.dropout_ratio = dropout_ratio

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)

        self.dropout = Dropout(dropout_ratio)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)  # (batch_size, seq_len, d_model)
        k = self.wk(k)  # (batch_size, seq_len, d_model)
        v = self.wv(v)  # (batch_size, seq_len, d_model)

        q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
        k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
        v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

        scaled_attention = ScaledDotProductAttention(self.dropout_ratio)(q, k, v, mask, training)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)
        output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)
        return output
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "dropout_ratio": self.dropout_ratio
        })
        return config



class TransformerMultiHeadAttentionEncoderLayer(Layer):
    def __init__(self, ff_dim, dropout_ratio, **kwargs):
        super(TransformerMultiHeadAttentionEncoderLayer, self).__init__(**kwargs)
        self.ff_dim = ff_dim
        self.dropout_ratio = dropout_ratio

    def build(self, input_shape):
        self.attention = MultiHeadAttention(d_model=input_shape[-1], num_heads=4, dropout_ratio=self.dropout_ratio)
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(self.dropout_ratio)
        self.dense1 = Dense(self.ff_dim, activation='gelu', kernel_regularizer=l2(0.01))
        self.dense2 = Dense(input_shape[-1], kernel_regularizer=l2(0.01))
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout2 = Dropout(self.dropout_ratio)
        super(TransformerMultiHeadAttentionEncoderLayer, self).build(input_shape)

    def call(self, inputs, training=False):
        seq_len, d_model = inputs.shape[1], inputs.shape[2]
        pos_encoding = positional_encoding(seq_len, d_model)
        x = inputs + pos_encoding

        attn_output = self.attention(x, x, x)  # Q, K, V are the same for self-attention
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "ff_dim": self.ff_dim,
            "dropout_ratio": self.dropout_ratio
        })
        return config
    
def transformer_model(input_shape, n_class, num_layers=2):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')
    
    # Feature extraction part
    x = TimeDistributed(Flatten())(model_input)
    x = TimeDistributed(Dense(128, activation='gelu'))(x)
    x = TimeDistributed(Dropout(0.5))(x)
    
    # Transformer encoder stack
    for _ in range(num_layers):
        x = TransformerMultiHeadAttentionEncoderLayer(ff_dim=256, dropout_ratio=0.1)(x)
    
    # Output part
    x = Flatten()(x)
    model_output = Dense(n_class, activation='softmax', name='name_model_output')(x)
    
    # Model compiling
    model = Model(inputs=model_input, outputs=model_output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.0005),
                  loss = loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_2 = transformer_model(input_shape=(36, 20, 20, 1), n_class=6, num_layers=2)

In [ ]:
LastValueQuantizer = tfmot.quantization.keras.quantizers.LastValueQuantizer
MovingAverageQuantizer = tfmot.quantization.keras.quantizers.MovingAverageQuantizer

# TimeDistributed Wrapper
class TimeDistributedQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    # No weights to quantize for TimeDistributed wrapper
    def get_weights_and_quantizers(self, layer):
        return []

    # Quantize the activations of the underlying layer if necessary
    def get_activations_and_quantizers(self, layer):
        # Assuming the underlying layer has an activation attribute
        if hasattr(layer.layer, 'activation'):
            return [(layer.layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
        return []

    # No weights to set for TimeDistributed wrapper
    def set_quantize_weights(self, layer, quantize_weights):
        pass

    # Set the quantized activations back to the underlying layer
    def set_quantize_activations(self, layer, quantize_activations):
        if hasattr(layer.layer, 'activation'):
            layer.layer.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}

# Transformer Attention Encoder Layer
class TransformerMultiHeadAttentionEncoderQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
   
  def get_weights_and_quantizers(self, layer):
    # Quantize the weights of the dense and layer normalization layers
    return [
        (layer.dense1.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
        (layer.dense2.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
    ]

  def get_activations_and_quantizers(self, layer):
    # Quantize the activations from the dense and layer normalization layers
    return [
        (layer.dense1.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False)),
        (layer.dense2.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False)),
    ]

  def set_quantize_weights(self, layer, quantize_weights):
    # Assign the quantized weights to the dense and layer normalization layers
    layer.dense1.kernel = quantize_weights[0]
    layer.dense2.kernel = quantize_weights[1]

  def set_quantize_activations(self, layer, quantize_activations):
    # Assign the quantized activations to the dense layers
    layer.dense1.activation = quantize_activations[0]
    layer.dense2.activation = quantize_activations[1]
    
  def get_output_quantizers(self, layer):
    # Define if you want to quantize the output of the layer
    return []

  def get_config(self):
    return {}


# Default Custom Config
class DefaultQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}



# Dense Layer
class DefaultDenseQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
   
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}    

class DenseQuantizeConfig(DefaultQuantizeConfig):

  #Change number of bits as needed
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False))]
  #Change number of bits as needed. Here same as default
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
  

# Activation in Dense layers
class DefaultActivationQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))] 

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}

class ActivationQuantizeConfig(DefaultActivationQuantizeConfig):

  def get_weights_and_quantizers(self, layer):
    return []
  def set_quantize_weights(self, layer, quantize_weights):
    return
  #Change number of bits as needed. Here same as default                                      
  def get_activations_and_quantizers (self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]


In [ ]:
quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
quantize_annotate_model = tfmot.quantization.keras.quantize_annotate_model
quantize_scope = tfmot.quantization.keras.quantize_scope

with quantize_scope(
      {'ScaledDotProductAttention': ScaledDotProductAttention,
       'MultiHeadAttention': MultiHeadAttention,
       'TimeDistributedQuantizeConfig':TimeDistributedQuantizeConfig,
       'TransformerMultiHeadAttentionEncoderLayer': TransformerMultiHeadAttentionEncoderLayer,
       'TransformerMultiHeadAttentionEncoderQuantizeConfig': TransformerMultiHeadAttentionEncoderQuantizeConfig,
       'ActivationQuantizeConfig': ActivationQuantizeConfig,
       'DenseQuantizeConfig': DenseQuantizeConfig,
}):
 
       def apply_quantization(layer):
          if isinstance(layer, tf.keras.layers.TimeDistributed):
             return tfmot.quantization.keras.quantize_annotate_layer(layer, TimeDistributedQuantizeConfig())
          elif isinstance(layer, tf.keras.layers.Flatten):
             return layer 
          elif isinstance(layer, tf.keras.layers.Dense):
            return tfmot.quantization.keras.quantize_annotate_layer(layer, DenseQuantizeConfig())
          elif isinstance(layer, tf.keras.layers.Dropout):
             return layer
          elif isinstance(layer, TransformerMultiHeadAttentionEncoderLayer):
             return tfmot.quantization.keras.quantize_annotate_layer(layer, TransformerMultiHeadAttentionEncoderQuantizeConfig()) 
          elif isinstance(layer, tf.keras.layers.Activation):
            return tfmot.quantization.keras.quantize_annotate_layer(layer, ActivationQuantizeConfig()) 
          elif isinstance(layer, tf.keras.layers.Dropout):
             return layer 
          return layer
        
       annotated_model = tf.keras.models.clone_model(
       model_2,clone_function= apply_quantization)
       quant_aware_model_2 = tfmot.quantization.keras.quantize_apply(annotated_model)
       quant_aware_model_2.summary()

In [ ]:
# Load the quantized model
interpreter = tf.lite.Interpreter(model_path='PTQ_teacher_model.tflite')
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Prepare the input data for the quantized model
def preprocess_input(data):
    # Ensure the input data is in the correct format and type
    input_shape = input_details[0]['shape']
    input_data = np.array(data, dtype=np.float32)
    return input_data

# Run Inference for the teacher_logits
# Precompute teacher logits for the entire training dataset
def precompute_teacher_logits(data):
    teacher_logits = []
    for sample in data:
        input_data = preprocess_input(sample)
        input_data = np.expand_dims(input_data, axis=0)  # Add batch dimension
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])
        teacher_logits.append(output_data)
    return np.array(teacher_logits)

# Precompute teacher logits for training and validation data
teacher_logits_train = precompute_teacher_logits(train_data)
teacher_logits_val = precompute_teacher_logits(val_data)


Knowledge Distillation (KD)

In [ ]:
# Define the distillation loss function
def distillation_loss(teacher_logits, student_logits, temperature):
    teacher_soft_logits = tf.nn.softmax(teacher_logits / temperature)
    student_soft_logits = tf.nn.softmax(student_logits / temperature)
    distillation_loss = tf.keras.losses.KLDivergence()(teacher_soft_logits, student_soft_logits)
    return distillation_loss

# Define the combined loss function
def combined_loss(teacher_logits, student_logits, true_labels, temperature, alpha):
    soft_loss = distillation_loss(teacher_logits, student_logits, temperature)
    hard_loss = tf.keras.losses.CategoricalCrossentropy()(true_labels, student_logits)
    return alpha * hard_loss + (1 - alpha) * soft_loss

# Custom training step using precomputed teacher logits
def train_step(data, label, teacher_logits, student_model, optimizer, temperature, alpha):
    with tf.GradientTape() as tape:
        student_logits = student_model(data, training=True)
        loss_value = combined_loss(teacher_logits, student_logits, label, temperature, alpha)
    gradients = tape.gradient(loss_value, student_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, student_model.trainable_variables))
    
    return loss_value

# Training hyperparameters
temperature = 15
alpha = 0.7
epochs = 40
batch_size = 24

# Define optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

# Training loop with precomputed teacher logits
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    for step, (data, label) in enumerate(training_generator):
        # Get precomputed teacher logits for the current batch
        batch_teacher_logits = teacher_logits_train[step * batch_size:(step + 1) * batch_size]
        loss_value = train_step(data, label, batch_teacher_logits, quant_aware_model_2, optimizer, temperature, alpha)
        if step % 100 == 0:
            print(f"Step {step}, Loss: {loss_value.numpy()}")
    
    val_loss = 0
    for step, (val_data, val_label) in enumerate(validation_generator):
        # Get precomputed teacher logits for the current batch
        batch_teacher_logits = teacher_logits_val[step * batch_size:(step + 1) * batch_size]
        student_logits = quant_aware_model_2(val_data, training=False)
        val_loss += combined_loss(batch_teacher_logits, student_logits, val_label, temperature, alpha).numpy()
    val_loss /= len(validation_generator)
    print(f"Validation Loss: {val_loss}")

    # Apply learning rate scheduler
    current_lr = optimizer.learning_rate.numpy()
    new_lr = scheduler(epoch, current_lr)
    optimizer.learning_rate.assign(new_lr)
    print(f"Learning rate for epoch {epoch+1} is {new_lr}")

    # Check for early stopping
    best_val_loss = 0
    patience_counter = 0
    if val_loss < best_val_loss:
         best_val_loss = val_loss
         patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= early_stopping.patience:
            print("Early stopping due to no improvement in validation loss.")

In [ ]:
# Qunatization of Student after applying KD
# Create a function that provides input samples
def representative_data_gen():
    batch_size = 32
    num_samples = data_train.shape[0]
    
    # Shuffle the data indices
    shuffled_indices = np.arange(num_samples)
    np.random.shuffle(shuffled_indices)
    
    # Iterate over the shuffled data, yielding batches
    for i in range(0, num_samples, batch_size):
        # Get the batch indices
        batch_indices = shuffled_indices[i:i + batch_size]
        # Yield a batch of data
        yield [data_train[batch_indices].astype(np.float32)]

# Quantization Aware Training
# Convert the model_1 to TensorFlow Lite format with 8-bit quantization
converter = tf.lite.TFLiteConverter.from_keras_model(quant_aware_model_2)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
converter.inference_input_type = tf.float32 # model is quantized to int8 but can accept input, output float32 datatype 
converter.inference_output_type = tf.float32 
converter.representative_dataset = representative_data_gen

# Disable experimental lowering of tensor list ops
converter._experimental_lower_tensor_list_ops = False

# Generate the TensorFlow Lite model with quantization
tflite_quantized_model = converter.convert()

# Save the quantized model to a file
with open('KD_QAT_student model.tflite', 'wb') as f:
    f.write(tflite_quantized_model)

print("Quantized model saved successfully.")

In [ ]:
# Student model testing time
def calculate_accuracy_and_loss(interpreter, data, labels, batch_size):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Resize the input tensor to accept the batch size
    input_shape = input_details[0]['shape']
    input_shape[0] = batch_size
    interpreter.resize_tensor_input(input_details[0]['index'], input_shape)
    interpreter.allocate_tensors()
    
    correct_predictions = 0
    total_predictions = 0
    total_loss = 0.0
    
    for i in range(0, data.shape[0], batch_size):
        # Prepare the batch of test examples and labels
        test_batch = data[i:i + batch_size].astype(np.float32)
        label_batch = labels[i:i + batch_size]
        
        # Check if the last batch is smaller than the batch size
        if test_batch.shape[0] < batch_size:
            # Resize the input tensor to the new batch size
            interpreter.resize_tensor_input(input_details[0]['index'], [test_batch.shape[0]] + list(input_shape[1:]))
            interpreter.allocate_tensors()
        
        # Run the model on the batch
        interpreter.set_tensor(input_details[0]['index'], test_batch)
        interpreter.invoke()
        
        # Get the results for the batch
        output_batch = interpreter.get_tensor(output_details[0]['index'])
        
        # Calculate loss (assuming categorical cross-entropy)
        loss = tf.keras.losses.categorical_crossentropy(label_batch, output_batch)
        total_loss += np.sum(loss)
        
        # Compare the results with the labels and count correct predictions
        predicted_labels = np.argmax(output_batch, axis=1)
        actual_labels = np.argmax(label_batch, axis=1)
        correct_predictions += (predicted_labels == actual_labels).sum()
        total_predictions += len(actual_labels)
        
        # If the input tensor was resized for the last batch, reset it to the original batch size
        if test_batch.shape[0] < batch_size:
            interpreter.resize_tensor_input(input_details[0]['index'], input_shape)
            interpreter.allocate_tensors()
    
    accuracy = float(correct_predictions) / total_predictions
    average_loss = total_loss / total_predictions
    return accuracy, average_loss

# Load the quantized TFLite model
interpreter = tf.lite.Interpreter(model_path='KD_QAT_student model.tflite')

# Define your datasets and labels
datasets = [data_test, data_test_1, data_test_2]
labels = [label_test, label_test_1, label_test_2]
batch_size = 24  # Define your desired batch size

# Calculate accuracy, loss, and evaluation time for each dataset
for i, (data, label) in enumerate(zip(datasets, labels)):
    start_time = time.time()
    accuracy, loss = calculate_accuracy_and_loss(interpreter, data, label, batch_size)
    end_time = time.time()
    evaluation_time = end_time - start_time
    print(f'Dataset {i+1} - Accuracy: {accuracy}, Loss: {loss}, Evaluation Time: {evaluation_time} seconds')